# cdc_loader — exploratory tests

Run each cell in order. All tables are written to `data/warehouse.duckdb`.

In [ ]:
import sys
sys.path.insert(0, '..')

import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s  %(message)s')

DATA = '../docs/sample-data'

## 1 — Load all CDC tables

In [ ]:
from src.ingest.cdc_loader import (
    load_transactions,
    load_reconciliation_runs,
    load_reconciliation_results,
    load_enterprise_company,
)

load_transactions([
    f'{DATA}/transactions_batch_1.parquet',
    f'{DATA}/transactions_batch_2.parquet',
])
load_reconciliation_runs(f'{DATA}/reconciliation_runs.parquet')
load_reconciliation_results(f'{DATA}/reconciliation_results.parquet')
load_enterprise_company(f'{DATA}/enterprise_company.parquet')

## 2 — Row counts per table

In [ ]:
from src.db import get_connection
import duckdb

conn = get_connection()

tables = [
    'raw_transactions',
    'raw_reconciliation_runs',
    'raw_reconciliation_results',
    'raw_enterprise_company',
]

rows = []
for t in tables:
    n = conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    rows.append({'table': t, 'rows': n})

conn.execute("SELECT unnest($1::VARCHAR[]) AS table", [tables]).df()  # just to show headers

import duckdb
duckdb.connect().execute(
    'SELECT * FROM (VALUES ' + ', '.join(f"('{r["table"]}', {r['rows']})" for r in rows) + ') t(table_name, row_count)'
).df()

## 3 — CDC: no deleted rows should survive

In [ ]:
for t in tables:
    deleted = conn.execute(f"SELECT COUNT(*) FROM {t} WHERE Op = 'D'").fetchone()[0]
    status = 'OK' if deleted == 0 else 'FAIL'
    print(f'[{status}] {t}: {deleted} deleted rows remaining')

## 4 — CDC: each PK appears exactly once (deduplication check)

In [ ]:
pk_map = {
    'raw_transactions':           'transaction_id',
    'raw_reconciliation_runs':    'id',
    'raw_reconciliation_results': 'id',
    'raw_enterprise_company':     'id',
}

for t, pk in pk_map.items():
    dups = conn.execute(f"""
        SELECT COUNT(*) FROM (
            SELECT {pk}, COUNT(*) AS cnt FROM {t} GROUP BY {pk} HAVING cnt > 1
        )
    """).fetchone()[0]
    status = 'OK' if dups == 0 else 'FAIL'
    print(f'[{status}] {t}: {dups} duplicate {pk}s')

## 5 — Schema drift: `payment_method` merged from batch_2 into raw_transactions

In [ ]:
cols = [r[0] for r in conn.execute('DESCRIBE raw_transactions').fetchall()]
print('Columns:', cols)
print('payment_method present:', 'payment_method' in cols)

# Rows from batch_1 (ids 1-742) have NULL payment_method; batch_2 may have a value
conn.execute("""
    SELECT
        CASE WHEN id <= 742 THEN 'batch_1' ELSE 'batch_2' END AS batch,
        COUNT(*)                                               AS total,
        COUNT(payment_method)                                  AS with_payment_method
    FROM raw_transactions
    GROUP BY batch
    ORDER BY batch
""").df()

## 6 — Idempotency: running twice must not change row counts

In [ ]:
before = {t: conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0] for t in tables}

load_transactions([
    f'{DATA}/transactions_batch_1.parquet',
    f'{DATA}/transactions_batch_2.parquet',
])
load_reconciliation_runs(f'{DATA}/reconciliation_runs.parquet')
load_reconciliation_results(f'{DATA}/reconciliation_results.parquet')
load_enterprise_company(f'{DATA}/enterprise_company.parquet')

after = {t: conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0] for t in tables}

for t in tables:
    status = 'OK' if before[t] == after[t] else 'FAIL'
    print(f'[{status}] {t}: {before[t]} → {after[t]}')

## 7 — raw_transactions sample

In [ ]:
conn.execute("SELECT * FROM raw_transactions ORDER BY id LIMIT 10").df()

## 8 — raw_enterprise_company sample (with updates applied)

In [ ]:
conn.execute("SELECT * FROM raw_enterprise_company ORDER BY id LIMIT 10").df()

In [ ]:
conn.close()